# Multi-View Gaze Dashboard

This notebook is an interactive figure finder for ADT gaze analysis. It is meant to help choose a sequence, frame window, and focus frame; a paper should only use a static export/screenshot of a selected view, not the widget controls.

The dashboard answers five linked questions:

1. **Local gaze:** how do CPF yaw/pitch change within the selected window?
2. **Motion:** are Scene gaze velocity and head rotation speed high in the same interval?
3. **Image space:** where does the gaze projection lie in RGB pixel coordinates?
4. **Object hit:** which frames have ray-box hits, and how far is the depth-defined gaze point from the box hit?
5. **3D context:** where are the object boxes, skeleton, head trajectory, gaze rays, and current hit point in Scene space?

Figure design notes are in `docs/multiview_gaze_dashboard_design.md`. The plotting logic lives in `visualization/multiview_dashboard.py`. Later, prediction results can be passed as extra gaze tracks with the same basic gaze columns and overlaid against GT.


In [3]:
from pathlib import Path
import sys
import importlib

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

import visualization.multiview_dashboard as multiview_dashboard
import visualization.scene_object_viewer as scene_object_viewer
importlib.reload(scene_object_viewer)
importlib.reload(multiview_dashboard)

from visualization.multiview_dashboard import build_multiview_dashboard, load_multiview_data
from visualization.scene_object_viewer import discover_scene_viewer_sequence_ids

REPORTS_DIR = Path("/mnt/d/SparseGaze/ADT-Gaze-structured")
sequence_ids = discover_scene_viewer_sequence_ids(REPORTS_DIR)
sequence_ids[:3], len(sequence_ids)


(['Apartment_release_decoration_skeleton_seq131_M1292',
  'Apartment_release_decoration_skeleton_seq132_M1292',
  'Apartment_release_decoration_skeleton_seq133_M1292'],
 34)

In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

_data_cache = {}
_prediction_tracks = {}

def get_data(sequence_id):
    if sequence_id not in _data_cache:
        _data_cache[sequence_id] = load_multiview_data(REPORTS_DIR, sequence_id)
    return _data_cache[sequence_id]

def load_prediction_csv(name, path):
    # Optional helper for later model outputs.
    # The CSV should contain query_timestamp_ns when possible, plus any of:
    # yaw_rad, pitch_rad, gaze_u_px, gaze_v_px, gaze_dir_scene_unit_x/y/z.
    _prediction_tracks[name] = pd.read_csv(path)

sequence_dropdown = widgets.Dropdown(
    options=sequence_ids,
    value=sequence_ids[0],
    description="sequence",
    layout=widgets.Layout(width="900px"),
)
start_input = widgets.IntText(value=0, description="start")
end_input = widgets.IntText(value=180, description="end")
stride_input = widgets.IntText(value=5, description="stride")
focus_input = widgets.IntText(value=-1, description="focus")
max_static_input = widgets.IntText(value=120, description="max static")

category_text = widgets.Text(value="", description="categories", layout=widgets.Layout(width="520px"))
exclude_category_text = widgets.Text(value="shelter", description="exclude", layout=widgets.Layout(width="520px"))

show_static_checkbox = widgets.Checkbox(value=True, description="static boxes")
show_dynamic_checkbox = widgets.Checkbox(value=True, description="dynamic boxes")
show_skeleton_checkbox = widgets.Checkbox(value=True, description="skeleton")
show_gaze_points_checkbox = widgets.Checkbox(value=True, description="gaze points")
show_hit_point_checkbox = widgets.Checkbox(value=True, description="hit point")
show_hit_object_checkbox = widgets.Checkbox(value=True, description="hit object")

gaze_length_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="ray len",
    readout_format=".1f",
)
gaze_scale_dropdown = widgets.Dropdown(
    options=["fixed", "depth"],
    value="fixed",
    description="ray scale",
)
render_button = widgets.Button(description="Render", button_style="primary")
output = widgets.Output()

def render(_=None):
    with output:
        output.clear_output(wait=True)
        data = get_data(sequence_dropdown.value)
        focus_frame = None if focus_input.value < 0 else int(focus_input.value)
        max_static = None if max_static_input.value <= 0 else int(max_static_input.value)
        fig = build_multiview_dashboard(
            data,
            start_frame=int(start_input.value),
            end_frame=int(end_input.value),
            stride=max(1, int(stride_input.value)),
            focus_frame=focus_frame,
            prediction_tracks=_prediction_tracks,
            show_static_objects=show_static_checkbox.value,
            show_dynamic_objects=show_dynamic_checkbox.value,
            max_static_objects=max_static,
            category_filter=category_text.value,
            exclude_category_filter=exclude_category_text.value,
            show_skeleton=show_skeleton_checkbox.value,
            show_gaze_points=show_gaze_points_checkbox.value,
            show_object_hits=show_hit_point_checkbox.value,
            show_hit_object=show_hit_object_checkbox.value,
            gaze_ray_length_m=gaze_length_slider.value,
            gaze_scale_mode=gaze_scale_dropdown.value,
        )
        display(fig)

render_button.on_click(render)

controls = widgets.VBox([
    sequence_dropdown,
    widgets.HBox([start_input, end_input, stride_input, focus_input, max_static_input]),
    widgets.HBox([category_text, exclude_category_text]),
    widgets.HBox([gaze_scale_dropdown, gaze_length_slider]),
    widgets.HBox([show_static_checkbox, show_dynamic_checkbox, show_skeleton_checkbox, show_gaze_points_checkbox, show_hit_point_checkbox, show_hit_object_checkbox]),
    render_button,
])

display(controls, output)
render()


Output()

## Prediction Track Hook

Later, after exporting model predictions to CSV, load them like this before
rendering:

```python
load_prediction_csv("model_a", "/path/to/model_a_gaze_predictions.csv")
load_prediction_csv("model_b", "/path/to/model_b_gaze_predictions.csv")
```

The prediction CSV should preferably include `query_timestamp_ns` so it can be
aligned to GT. Supported overlay columns are:

- `yaw_rad`, `pitch_rad`
- `gaze_u_px`, `gaze_v_px`
- `gaze_dir_scene_unit_x`, `gaze_dir_scene_unit_y`, `gaze_dir_scene_unit_z`

This is intentionally a visualization hook, not yet a full quantitative
prediction evaluator.
